##3-level Stigma Classification

In [ ]:
pip install --upgrade google-genai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.8/728.8 kB 12.4 MB/s eta 0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.63.0
    Uninstalling google-genai-1.63.0:
      Successfully uninstalled google-genai-1.63.0


In [ ]:
import os
from google.colab import auth

auth.authenticate_user()
os.environ["GOOGLE_CLOUD_API_KEY"] = "YOUR_API_KEY"

In [ ]:
from google import genai


PROJECT_ID = "gen-lang-client-0150300232"
LOCATION = "global"  # Vertex AI requires a specific region

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Hello!"
)

print(response.text)

Hi there! How can I help you today?



In [ ]:
import os, time, json, re
import pandas as pd
import openai
from openai import OpenAI
from typing import List, Dict, Any
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from threading import Lock

In [ ]:
def safe_json_load(text: str) -> Dict[str, Any]:
    if not isinstance(text, str):
        return {}
    text = text.strip()

    # direct parse
    try:
        return json.loads(text)
    except Exception:
        pass

    # extract first JSON block
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return {}

    return {}

def chat_to_text(client, model: str, messages: List[Dict[str, str]], max_tokens: int = 300) -> str:
    for _ in range(3):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=0,
                max_tokens=max_tokens,
                messages=messages
            )
            return resp.choices[0].message.content.strip()
        except openai.RateLimitError:
            time.sleep(retry_wait_time)
        except Exception:
            time.sleep(retry_wait_time)
    return ""

def chat_to_json(client, model: str, messages: List[Dict[str, str]], max_tokens: int = 300) -> Dict[str, Any]:
    txt = chat_to_text(client, model, messages, max_tokens=max_tokens)
    return safe_json_load(txt)

###Level-1

In [ ]:
SYSTEM_L1 = """
You are a strict classifier.

TASK (Level 1):
Decide whether the post is APPLICABLE for sexual-violence stigma annotation.

LABELS:
- Applicable: The post explicitly discusses sexual violence, sexual assault, rape,
  molestation, sexual abuse, coercion, consent violations, or reporting/disclosure
  of such experiences (including childhood or partner sexual abuse).
- Not Applicable: The post does NOT involve sexual violence. This includes general
  mental health issues, relationship problems, harassment without sexual violence,
  or content that can be generalized outside an SV context.

Return ONLY valid JSON in this exact format:
{
  "label": "Applicable" or "Not Applicable",
  "reason": "one short sentence",
  "evidence": "3–25 word direct snippet from the post, or empty string if none"
}

RULES:
- Choose exactly ONE label.
- Evidence must be copied verbatim from the post (no paraphrasing).
"""

In [ ]:
def run_level1(text: str, client, model: str = MODEL_NAME) -> Dict[str, Any]:
    msgs = [
        {"role": "system", "content": SYSTEM_L1},
        {"role": "user", "content": str(text)}
    ]
    return chat_to_json(client, model, msgs, max_tokens=220)

def predict_level1(text: str) -> Dict[str, Any]:
    if text is None or not str(text).strip():
        return {"label": "Not Applicable", "reason": "empty text", "evidence": ""}
    return run_level1(text, client, MODEL_NAME)

In [ ]:
from tqdm import tqdm
import pandas as pd

L1_INPUT_PATH = "content/Sexual Violence sampled data - Level_1_annotation_no_labels.csv"
L1_OUTPUT_PATH = "content/level1_predictions.csv"

TEXT_COL = "body"

df_l1 = pd.read_csv(L1_INPUT_PATH)

# enable tqdm for pandas
tqdm.pandas()

# run Level 1 with progress bar
df_l1["l1_json"] = df_l1[TEXT_COL].progress_apply(predict_level1)

# unpack outputs
df_l1["l1_label"] = df_l1["l1_json"].apply(
    lambda j: j.get("label", "Not Applicable") if isinstance(j, dict) else "Not Applicable"
)
df_l1["l1_reason"] = df_l1["l1_json"].apply(
    lambda j: j.get("reason", "") if isinstance(j, dict) else ""
)
df_l1["l1_evidence"] = df_l1["l1_json"].apply(
    lambda j: j.get("evidence", "") if isinstance(j, dict) else ""
)

# save results
df_l1.to_csv(L1_OUTPUT_PATH, index=False)
print("Saved Level 1 predictions:", L1_OUTPUT_PATH)


In [ ]:
L1_GT_PATH = "content/Sexual Violence sampled data - Level_1_annotation.csv"
GT_L1_COL = "Tags"

df_gt_l1 = pd.read_csv(L1_GT_PATH)

ID_COL = "Post ID"

if ID_COL in df_l1.columns and ID_COL in df_gt_l1.columns:
    merged = df_gt_l1[[ID_COL, GT_L1_COL]].merge(df_l1[[ID_COL, "l1_label"]], on=ID_COL, how="inner")
else:

    merged = pd.DataFrame({
        "gt": df_gt_l1[GT_L1_COL].astype(str).str.strip(),
        "pred": df_l1["l1_label"].astype(str).str.strip()
    })

merged["gt"] = merged[GT_L1_COL].astype(str).str.strip() if GT_L1_COL in merged.columns else merged["gt"]
merged["pred"] = merged["l1_label"].astype(str).str.strip() if "l1_label" in merged.columns else merged["pred"]

acc = accuracy_score(merged["gt"], merged["pred"])
p, r, f1, _ = precision_recall_fscore_support(
    merged["gt"],
    merged["pred"],
    average="binary",
    pos_label="Applicable"
)

print("LEVEL 1 Metrics (Applicable = positive)")
print("Accuracy:", acc)
print("Precision:", p)
print("Recall:", r)
print("F1:", f1)
print("Compared rows:", len(merged))

In [ ]:
df_compare = merged.copy()

df_compare["match"] = df_compare["gt"] == df_compare["pred"]

df_compare_view = df_compare[[ID_COL, "gt", "pred", "match"]]

df_compare_view.head(15)


### Level-2

In [ ]:
SYSTEM_L2 = """
You are a strict classifier.

TASK (Level 2):
Decide whether the post contains STIGMA about sexual violence (SV).

Return:
- "Stigma" if ANY stigma cues exist
- "No Stigma" if SV is discussed but no stigma cues exist

Stigma cues include ANY of:
- Experienced: blamed/disbelieved/minimized/shamed after disclosure/hostile norms in school/town/culture/religion/online forum
- Internalized: self-blame, shame, guilt, dirty, broken, damaged
- Anticipated: fear of judgment/blame/disbelief if disclosing
- Structural: institutional barriers (police/university/hospital/courts/policies)

Return ONLY valid JSON:
{
  "label": "Stigma" or "No Stigma",
  "reason": "one short sentence",
  "evidence": "3–25 word direct snippet, or empty string"
}
"""

In [ ]:
def run_level2(text: str, client, model: str = MODEL_NAME) -> Dict[str, Any]:
    msgs = [{"role": "system", "content": SYSTEM_L2},
            {"role": "user", "content": str(text)}]
    return chat_to_json(client, model, msgs, max_tokens=220)

def predict_level2(text: str) -> Dict[str, Any]:
    if text is None or not str(text).strip():
        return {"label": "No Stigma", "reason": "empty text", "evidence": ""}
    return run_level2(text, client, MODEL_NAME)

L2_INPUT_PATH = "content/Sexual Violence sampled data - Level_2_annotation_no_labels.csv"
L2_OUTPUT_PATH = "content/level2_predictions.csv"
df_l2 = pd.read_csv(L2_INPUT_PATH)

df_l2["l2_json"] = df_l2[TEXT_COL].apply(predict_level2)
df_l2["l2_label"] = df_l2["l2_json"].apply(lambda j: j.get("label", "No Stigma") if isinstance(j, dict) else "No Stigma")
df_l2["l2_reason"] = df_l2["l2_json"].apply(lambda j: j.get("reason", "") if isinstance(j, dict) else "")
df_l2["l2_evidence"] = df_l2["l2_json"].apply(lambda j: j.get("evidence", "") if isinstance(j, dict) else "")

df_l2.to_csv(L2_OUTPUT_PATH, index=False)
print("Saved Level 2 predictions:", L2_OUTPUT_PATH)

In [ ]:
L2_GT_PATH = "content/Sexual Violence sampled data - Level_2_annotation.csv"
GT_L2_COL = "Tags"

df_gt_l2 = pd.read_csv(L2_GT_PATH)

if ID_COL in df_l2.columns and ID_COL in df_gt_l2.columns:
    merged2 = df_gt_l2[[ID_COL, GT_L2_COL]].merge(df_l2[[ID_COL, "l2_label"]], on=ID_COL, how="inner")
    gt = merged2[GT_L2_COL].astype(str).str.strip()
    pred = merged2["l2_label"].astype(str).str.strip()
else:
    gt = df_gt_l2[GT_L2_COL].astype(str).str.strip()
    pred = df_l2["l2_label"].astype(str).str.strip()

acc2 = accuracy_score(gt, pred)
p2, r2, f12, _ = precision_recall_fscore_support(
    gt, pred,
    average="binary",
    pos_label="Stigma"
)

print("LEVEL 2 Metrics (Stigma = positive)")
print("Accuracy:", acc2)
print("Precision:", p2)
print("Recall:", r2)
print("F1:", f12)
print("Compared rows:", len(gt))

In [ ]:
df_check_l2 = merged2.rename(columns={
    GT_L2_COL: "GT_Label",
    "l2_label": "Pred_Label"
})

df_check_l2.head(20)

### Level-3

In [ ]:
import json, re, time
from typing import Dict, Any, List, Tuple
import pandas as pd

ID_COL = "Post ID"     # column in both CSVs
TEXT_COL = "body"      # column in both CSVs
TAGS_COL = "Tags"      # ONLY exists in the labeled CSV

FINE_LABELS = ["Experienced", "Internalized", "Anticipated", "Structural"]

# ---- Files ----
# Posts you want to label (no tags)
L3_INPUT_PATH  = "/content/Sexual Violence sampled data - Level_3_annotation_no_labels (1).csv"

# Labeled pool used for neighbors (HAS Tags)
L3_LABELED_POOL_PATH = "/content/Sexual Violence sampled data - Level_3_annotation (1).csv"

# Similarity mapping (computed earlier)
SIM_JSON_PATH = "/content/similar_posts_k3_75_5.json"

# Output
L3_OUTPUT_PATH = "/content/L3_predictions.csv"

In [ ]:
# ======================================================
# CELL 4 — gemini_chat_to_text / safe_json_load / gemini_chat_to_json  ✅ COPY-PASTE
# Robust to: code fences, trailing commas, True/False/None, extra text,
# greedy-brace issues, occasional single-quote dicts, empty outputs
# ======================================================

from typing import List, Dict, Any
import json
import re
import time

# -------------------------------
# Gemini call wrapper
# -------------------------------
def gemini_chat_to_text(
    client,
    model: str,
    messages: List[Dict[str, str]],
    max_tokens: int = 500,
    max_retries: int = 3,
    sleep_s: float = 1.2,
    temperature: float = 0.0,
) -> str:
    """
    Calls Gemini via OpenAI-compatible client.chat.completions.create and returns text.
    Retries on empty content or API exceptions.
    """
    last_err = None
    last_txt = ""

    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=messages,
            )
            txt = (resp.choices[0].message.content or "").strip()
            last_txt = txt

            # Retry if empty
            if not txt:
                last_err = ValueError(f"Empty model content (attempt {attempt}).")
                time.sleep(sleep_s)
                continue

            return txt

        except Exception as e:
            last_err = e
            time.sleep(sleep_s)

    print("❌ gemini_chat_to_text failed:", last_err)
    if last_txt:
        print("   Last text head:", last_txt[:220])
        print("   Last text tail:", last_txt[-220:])
    return ""


# -------------------------------
# Robust JSON extraction + light repairs
# -------------------------------
def safe_json_load(text: str) -> Dict[str, Any]:
    """
    Robust JSON extraction + light repairs.
    Returns {} if unable to parse a JSON object.
    """
    if not isinstance(text, str):
        return {}
    t = text.strip()
    if not t:
        return {}

    # Strip markdown fences if present
    t = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"^```\s*", "", t).strip()
    t = re.sub(r"\s*```$", "", t).strip()

    # 1) Direct parse
    try:
        obj = json.loads(t)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        pass

    # 2) Extract first JSON object block (NON-GREEDY)
    m = re.search(r"\{.*?\}", t, flags=re.DOTALL)
    if not m:
        return {}

    blob = m.group(0).strip()

    # Light repairs:
    # - trailing commas
    blob = re.sub(r",\s*}", "}", blob)
    blob = re.sub(r",\s*]", "]", blob)

    # - Python booleans / None -> JSON
    blob = re.sub(r"\bTrue\b", "true", blob)
    blob = re.sub(r"\bFalse\b", "false", blob)
    blob = re.sub(r"\bNone\b", "null", blob)

    # - occasional single-quote dicts (heuristic)
    #   Only apply if it looks like it's using single quotes consistently
    if blob.startswith("{") and "'" in blob and '"' not in blob:
        blob = blob.replace("'", '"')

    try:
        obj = json.loads(blob)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}


# -------------------------------
# Chat -> JSON with schema validation + retries
# -------------------------------
def gemini_chat_to_json(
    client,
    model: str,
    messages: List[Dict[str, str]],
    max_tokens: int = 520,
    max_retries: int = 3,
    sleep_s: float = 1.2,
    temperature: float = 0.0,
    required_keys: set = None,
) -> Dict[str, Any]:
    """
    Calls gemini_chat_to_text and parses JSON.
    Retries if JSON parse fails or required keys are missing / wrong types.

    Default expected schema:
      { "reasoning": str, "present": bool, "evidence": str }
    """
    if required_keys is None:
        required_keys = {"reasoning", "present", "evidence"}

    last_txt = ""
    last_err = None

    for attempt in range(1, max_retries + 1):
        txt = gemini_chat_to_text(
            client=client,
            model=model,
            messages=messages,
            max_tokens=max_tokens,
            max_retries=1,      # outer loop controls retries
            sleep_s=sleep_s,
            temperature=temperature,
        )
        last_txt = txt

        parsed = safe_json_load(txt)

        # Retry if parse failed
        if not parsed:
            last_err = ValueError(
                f"JSON parse failed (attempt {attempt}). Raw head: {txt[:220]}"
            )
            time.sleep(sleep_s)
            continue

        # Validate required keys
        if not required_keys.issubset(parsed.keys()):
            last_err = ValueError(
                f"JSON missing keys (attempt {attempt}). Parsed keys: {list(parsed.keys())}"
            )
            time.sleep(sleep_s)
            continue

        # Validate types
        if not isinstance(parsed.get("reasoning"), str):
            last_err = ValueError(f"Bad type for 'reasoning' (attempt {attempt}).")
            time.sleep(sleep_s)
            continue
        if not isinstance(parsed.get("present"), bool):
            last_err = ValueError(f"Bad type for 'present' (attempt {attempt}).")
            time.sleep(sleep_s)
            continue
        if not isinstance(parsed.get("evidence"), str):
            last_err = ValueError(f"Bad type for 'evidence' (attempt {attempt}).")
            time.sleep(sleep_s)
            continue

        return parsed

    print("❌ gemini_chat_to_json failed:", last_err)
    if last_txt:
        print("   Last raw head:", last_txt[:220])
        print("   Last raw tail:", last_txt[-220:])
    return {}

In [ ]:
# ==========================================
# CELL 5 — Canonicalize / Validate L3 Output ✅ COPY-PASTE
# ==========================================

from typing import Dict, Any, List

# Use your stigma labels as the ONLY canonical label set
L3_LABELS: List[str] = list(L3_SINGLE_LABEL_DEFINITIONS.keys())  # ["Experienced","Internalized","Anticipated","Structural"]

def canon_l3_output(j: Dict[str, Any]) -> Dict[str, Any]:
    """
    Ensures:
    - label_presence has all keys in L3_LABELS
    - labels aligns with label_presence (lp is source of truth)
    - evidence exists only for present labels
    - coerces common model mistakes: "true"/"false" strings -> bool
    """
    if not isinstance(j, dict):
        j = {}

    lp = j.get("label_presence", {})
    if not isinstance(lp, dict):
        lp = {}

    def _to_bool(v: Any) -> bool:
        if v is True:
            return True
        if v is False or v is None:
            return False
        if isinstance(v, str):
            s = v.strip().lower()
            if s in {"true", "yes", "y", "1"}:
                return True
            if s in {"false", "no", "n", "0", ""}:
                return False
        if isinstance(v, (int, float)):
            return bool(v)
        return False

    # fill missing keys + coerce to bool
    lp2: Dict[str, bool] = {}
    for k in L3_LABELS:
        lp2[k] = _to_bool(lp.get(k, False))

    # labels: enforce list + canonicalize + SOURCE OF TRUTH = lp2
    labels = [k for k, v in lp2.items() if v]

    # evidence: dict only; keep only evidence for present labels and non-empty strings
    ev = j.get("evidence", {})
    if not isinstance(ev, dict):
        ev = {}

    ev2: Dict[str, str] = {}
    for k in labels:
        val = ev.get(k, "")
        s = str(val).strip() if val is not None else ""
        if s:
            ev2[k] = s

    return {"label_presence": lp2, "labels": labels, "evidence": ev2}

In [ ]:
# from typing import Dict, List, Tuple, Any
# import os, json, re
# import pandas as pd

# # ======================================================
# # L3 STIGMA PROMPTING (K-SHOT NEIGHBORS)
# # COPY-PASTE CELL (Fixed tag parsing + consistent fewshot text)
# #
# # Assumes df_in and df_pool are the SAME ROW ORDER / SAME IDS
# # df_in: input posts (no Tag col needed)
# # df_pool: same dataset but HAS Tag column for gold labels
# # SIM_JSON_PATH maps target_idx -> [[neighbor_idx, score], ...]
# #
# # REQUIRED VARIABLES to set before running:
# #   L3_INPUT_PATH = "...csv"
# #   L3_LABELED_POOL_PATH = "...csv"
# #   SIM_JSON_PATH = "...json"
# #   TEXT_COL = "post_text"   (or your column)
# #   TAGS_COL = "Tag"
# # ======================================================

# # ------------------------------------------------------
# # 1) DEFINITIONS (Single-label evaluation per call)
# # ------------------------------------------------------
# L3_SINGLE_LABEL_DEFINITIONS: Dict[str, str] = {
#    "Experienced": (
#     "Any stigmatizing reactions, attitudes, or behaviors directed toward the survivor by other people. "
#     "This includes direct negative responses after disclosure (blame, disbelief, minimization, shaming, dismissal), "
#     "social rejection, abandonment, gossip, reputation damage, or people defending/choosing the perpetrator over the survivor. "
#     "Also includes hostile stigma norms described at the group level (e.g., school, workplace, culture, online community). "
#     "NOTE: Community-level stigma is merged into Experienced."
#     ),
#    "Internalized": (
#        "Survivor expresses self-blame, shame, guilt, or feeling dirty/broken "
#        "because of the assault."
#    ),
#    "Anticipated": (
#     "Fear or expectation of being judged, blamed, disbelieved, shamed, rejected, or punished IF they disclose "
#     "or seek help. This includes staying silent, avoiding reporting, or hesitating to tell others because of "
#     "expected negative reactions from friends, family, authorities, institutions, or the community. "
#     "The fear must be tied to anticipated social consequences of disclosure or help-seeking, rather than "
#     "fear related only to physical danger, the perpetrator, trauma symptoms, or general anxiety."
#   ),

#    "Structural": (
#        "Institutional or systemic barriers (police, hospital, university, courts, policy) "
#        "that obstruct help or justice."
#    ),
# }

# # Canonical label list
# L3_LABELS: List[str] = list(L3_SINGLE_LABEL_DEFINITIONS.keys())
# _CANON_L3 = {lab.lower(): lab for lab in L3_LABELS}

# # -------------------------------
# # FEWSHOT DISPLAY MODE
# # -------------------------------
# FEWSHOT_MODE = "snippet"      # "full" | "truncate" | "snippet"
# FEWSHOT_TRUNC_CHARS = 1200   # used if mode="truncate"

# def _sanitize_fewshot_snippet(text: str) -> str:
#     if not text:
#         return ""
#     s = str(text)
#     repl = {
#         r"\brape\b": "sexual assault",
#         r"\braped\b": "sexually assaulted",
#         r"\braping\b": "sexually assaulting",
#         r"\bmolest(ed|ation)?\b": "sexual abuse",
#         r"\bforced\b": "coerced",
#         r"\bpenetrat(e|ed|ion)\b": "sexual act",
#     }
#     for pat, rep in repl.items():
#         s = re.sub(pat, rep, s, flags=re.IGNORECASE)
#     return s

# # -------------------------------
# # Tag parsing (FIXED: no FINE_LABELS, case-insensitive, canonical L3 labels)
# # -------------------------------
# def parse_tags(tag_cell: Any) -> List[str]:
#     if tag_cell is None:
#         return []
#     s = str(tag_cell).strip()
#     if not s or s.lower() == "nan":
#         return []
#     parts = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]

#     out: List[str] = []
#     for p in parts:
#         key = p.lower().strip()
#         if key in _CANON_L3:
#             out.append(_CANON_L3[key])

#     # dedupe preserve order
#     seen = set()
#     deduped = []
#     for x in out:
#         if x not in seen:
#             deduped.append(x)
#             seen.add(x)
#     return deduped

# # -------------------------------
# # Snippet extraction for fewshots (keyword-guided)
# # -------------------------------
# def _extract_snippet(post_text: str, lab: str, max_words: int = 28) -> str:
#     text = str(post_text or "").strip()
#     if not text:
#         return ""

#     sents = re.split(r'(?<=[.!?])\s+', text)
#     sents = [s.strip() for s in sents if s.strip()]

#     lab_kw = {
#         "Experienced": [
#             "blame","blamed","disbelieve","didn't believe","minimize","shame","shamed","dismiss",
#             "mock","laughed","fault","liar","lying","gossip","rumor","reputation","abandon","rejected"
#         ],
#         "Internalized": [
#             "i feel","guilty","ashamed","dirty","tainted","worthless","ruined","broken",
#             "my fault","i blame myself","i hate myself"
#         ],
#         "Anticipated": [
#             "i didn't tell","i never told","afraid","scared","worried","people would","no one would",
#             "nobody would","wouldn't believe","judge","reputation","get in trouble","make a scene",
#             "they won't believe me","they'll think i'm lying"
#         ],
#         "Structural": [
#             "police","title ix","hr","hospital","court","judge","report","investigation",
#             "did nothing","refused","dismissed","ignored","no jurisdiction","lack of evidence","denied","blocked"
#         ],
#     }.get(lab, [])

#     def score(sent: str) -> int:
#         low = sent.lower()
#         return sum(1 for k in lab_kw if k in low)

#     scored = [(score(s), s) for s in sents]
#     scored.sort(key=lambda x: (x[0], -len(x[1])), reverse=True)
#     best = scored[0][1] if scored else text

#     best = _sanitize_fewshot_snippet(best)
#     words = best.split()
#     if len(words) > max_words:
#         best = " ".join(words[:max_words]) + "…"
#     return best

# def _fewshot_post_for_prompt(post_text: str, lab: str) -> str:
#     post_text = str(post_text or "").strip()
#     if FEWSHOT_MODE == "snippet":
#         return _extract_snippet(post_text, lab, max_words=28)
#     if FEWSHOT_MODE == "full":
#         return _sanitize_fewshot_snippet(post_text)
#     if FEWSHOT_MODE == "truncate":
#         post_text = _sanitize_fewshot_snippet(post_text)
#         if len(post_text) <= FEWSHOT_TRUNC_CHARS:
#             return post_text
#         return post_text[:FEWSHOT_TRUNC_CHARS] + "\n...[TRUNCATED]..."
#     return _extract_snippet(post_text, lab, max_words=28)

# # -------------------------------
# # Load data (uses your paths)
# # -------------------------------
# assert os.path.exists(L3_INPUT_PATH), L3_INPUT_PATH
# assert os.path.exists(L3_LABELED_POOL_PATH), L3_LABELED_POOL_PATH
# assert os.path.exists(SIM_JSON_PATH), SIM_JSON_PATH

# df_in = pd.read_csv(L3_INPUT_PATH)
# df_pool = pd.read_csv(L3_LABELED_POOL_PATH)

# with open(SIM_JSON_PATH, "r") as f:
#     SIM_MAP: Dict[str, List[List[Any]]] = json.load(f)

# print("✅ df_in:", df_in.shape, "| df_pool:", df_pool.shape, "| SIM keys:", len(SIM_MAP))

# if TEXT_COL not in df_in.columns:
#     raise ValueError(f"TEXT_COL '{TEXT_COL}' not found in df_in.")
# if TEXT_COL not in df_pool.columns:
#     raise ValueError(f"TEXT_COL '{TEXT_COL}' not found in df_pool.")
# if TAGS_COL not in df_pool.columns:
#     raise ValueError(f"TAGS_COL '{TAGS_COL}' not found in df_pool (needed for fewshots).")

# # -------------------------------
# # Neighbor fetch
# # -------------------------------
# def get_neighbor_records(target_idx: int, k: int = 3, min_score: float = None) -> List[Dict[str, Any]]:
#     raw = (SIM_MAP.get(str(target_idx), []) or [])[:k]
#     out = []
#     for pair in raw:
#         if not isinstance(pair, (list, tuple)) or len(pair) < 1:
#             continue
#         pool_idx = int(pair[0])
#         score = float(pair[1]) if len(pair) > 1 else 1.0
#         if min_score is not None and score < float(min_score):
#             continue
#         if pool_idx < 0 or pool_idx >= len(df_pool):
#             continue

#         post_text = "" if pd.isna(df_pool.loc[pool_idx, TEXT_COL]) else str(df_pool.loc[pool_idx, TEXT_COL])
#         tags = parse_tags(df_pool.loc[pool_idx, TAGS_COL])
#         out.append({"pool_idx": pool_idx, "score": score, "text": post_text, "tags": tags})
#     return out

# # -------------------------------
# # Fewshot builder (FIXED: keep shown snippet consistent with JSON)
# # -------------------------------
# def build_labelwise_fewshots_for_target(
#     target_idx: int,
#     lab: str,
#     k: int = 3,
#     min_score: float = None,
# ) -> List[Tuple[str, str]]:
#     """
#     Returns List[(shown_post, answer_json_str)] for THIS label only.
#     Includes neighbors even if present=False (negative examples).
#     """
#     neighbors = get_neighbor_records(target_idx, k=k, min_score=min_score)
#     fewshots: List[Tuple[str, str]] = []

#     for nb in neighbors:
#         nb_text = nb["text"]
#         nb_tags = nb["tags"]
#         is_pos = (lab in nb_tags)

#         shown = _fewshot_post_for_prompt(nb_text, lab)

#         ex = {
#             "present": bool(is_pos),
#             "evidence": _extract_snippet(nb_text, lab, max_words=20).replace("\n", " ").strip() if is_pos else ""
#         }
#         fewshots.append((shown, json.dumps(ex, ensure_ascii=False)))

#     return fewshots

# # -------------------------------
# # Prompt builder
# # -------------------------------
# def build_l3_system_prompt(
#     lab: str,
#     target_idx: int,
#     k_neighbors: int = 3,
#     max_examples: int = 3,
#     min_score: float = None,
# ) -> str:
#     definition = L3_SINGLE_LABEL_DEFINITIONS[lab].strip()

#     system_prompt = f"""
# You are a JSON-only annotator specializing in social stigma detection.
# You will evaluate ONE label at a time.

# GLOBAL RULES:
# - Use ONLY the post text. Do NOT infer hidden meanings.
# - Decide strictly for the label under test: "{lab}".
# - "present" is true ONLY if there is explicit evidence matching the label definition and boundary rules.
# - The "evidence" MUST be a verbatim snippet from the post proving the label (3–25 words).
# - If evidence is weak/ambiguous OR you cannot quote clear evidence, set "present": false.
# - Do NOT wrap the JSON in markdown fences (no ```json).
# - Return ONLY valid JSON.

# LABEL UNDER TEST:
# {lab}: {definition}

# LABEL-SPECIFIC BOUNDARY RULES:
# """.strip()

#     if lab == "Experienced":
#       system_prompt += """
#   Experienced = stigma coming FROM OTHER PEOPLE toward the survivor.

#   TRUE when:
#   - Someone blames, doubts, minimizes, dismisses, mocks, or shames the survivor
#   - Someone defends, supports, or sides with the perpetrator instead of the survivor
#   - Someone withdraws support, abandons, avoids, or rejects the survivor after learning about the assault
#   - Someone gossips, spreads rumors, or damages the survivor's reputation
#   - A group/community expresses stigmatizing attitudes about survivors (school, work, culture, online)

#   FALSE when:
#   - Only the survivor's own guilt/shame is described (Internalized)
#   - The problem is emotional pain, conflict, or lack of support WITHOUT stigma or judgment
#   - Someone simply fails to help but does not judge, blame, reject, or stigmatize

#   Evidence must clearly show:
#   1) WHO is acting (friend, family, partner, community, people online, etc.)
#   2) WHAT stigmatizing action or attitude occurred
#   """.strip()


#     elif lab == "Internalized":
#         system_prompt += """
# - TRUE only if the survivor expresses first-person self-blame/shame/guilt/dirty/broken feelings.
# - FALSE if blame/shame comes from other people (Experienced).
# """.strip()

#     elif lab == "Anticipated":
#         system_prompt += """
#     Anticipated:

#     TRUE if the text shows expected negative social/institutional reactions tied to disclosure or help-seeking.
#     Require BOTH:

#     1) Disclosure/help-seeking is mentioned explicitly OR implicitly.
#       - Explicit: tell/told, disclose, speak up, open up, report, police, Title IX, HR, counselor/therapist, court, hotline, seek help, reach out.
#       - Implicit (counts only if clearly about disclosure/help-seeking): "never told anyone", "kept it to myself", "stayed silent",
#         "didn't report", "couldn't say anything", "can't tell anyone", "afraid to speak up", "didn't get help".

#     2) Expected negative reaction from others is stated or strongly implied:
#       judged, blamed, disbelieved, shamed, rejected, punished, dismissed/minimized, laughed at, gossiped about,
#       "they won't believe me", "they'll think I'm lying", "they'll say it's my fault", "people will judge me".

#     FALSE if fear is only about:
#     - physical danger, the perpetrator, retaliation by the perpetrator, trauma symptoms, or general anxiety,
#     and there is no link to disclosure/help-seeking + others' reactions.

#     Evidence:
#     - Quote the phrase showing the expected reaction AND the disclosure/help-seeking mention (explicit or implicit).
#     """.strip()



#     elif lab == "Structural":
#        system_prompt += """
# Structural:
# - TRUE if an institution/system/process blocks help or justice.
# - Institution may be police, hospital, court, Title IX, HR, work investigation, legal process.
# - MUST include an obstruction or failure.
#  Barrier verbs: did nothing, refused, ignored, dismissed, denied, blocked, no jurisdiction.
# - FALSE if an institution is mentioned with no barrier.
# """.strip()

#     fewshots = build_labelwise_fewshots_for_target(
#         target_idx=target_idx,
#         lab=lab,
#         k=k_neighbors,
#         min_score=min_score,
#     )

#     system_prompt += f"""

# HOW TO USE THE EXAMPLES BELOW:
# - The following are labeled examples of posts similar to the one you will evaluate.
# - Each example shows how the label "{lab}" can be present (true) or absent (false).
# - Learn the decision pattern from these examples:
#   • What kind of language counts as evidence
#   • When the label should be false even if the story involves assault
# - Use these examples as guidance for both reasoning style and decision boundaries.
# """

#     system_prompt += "\n\nFEW-SHOT EXAMPLES:\n"
#     for i, (shown_post, ans_json) in enumerate(fewshots[:max_examples], start=1):
#         system_prompt += f"""
# EXAMPLE {i}
# NEIGHBOR POST ({FEWSHOT_MODE}):
# {shown_post}

# ANSWER JSON:
# {ans_json}
# """.rstrip()

#     system_prompt += f"""

# NOW LABEL THIS POST for the "{lab}" category ONLY.

# Return ONLY valid JSON in exactly this format:
# {{
#   "reasoning": "1–3 sentences. Name the Actor and the Action/Fear for {lab}.",
#   "present": true or false,
#   "evidence": "verbatim snippet if true (3–25 words), else empty string"
# }}
# """.strip()

#     return system_prompt

# def build_final_prompt_for_debug(system_prompt: str, target_post_text: str) -> str:
#     target_post_text = "" if target_post_text is None else str(target_post_text)
#     return (
#         f"{system_prompt}\n\n"
#         f"POST:\n{target_post_text}\n\n"
#         "FINAL INSTRUCTIONS:\n"
#         "Output ONLY a single JSON object (no markdown, no extra text).\n"
#         "Use lowercase true/false.\n"
#         "Include exactly these keys: reasoning, present, evidence.\n"
#     )

# # -------------------------------
# # DEBUG PRINT: show prompt for one target+label
# # -------------------------------
# target_idx_debug = 0
# lab_debug = "Anticipated"

# target_text_debug = "" if pd.isna(df_in.loc[target_idx_debug, TEXT_COL]) else str(df_in.loc[target_idx_debug, TEXT_COL])

# sys_prompt = build_l3_system_prompt(
#     lab=lab_debug,
#     target_idx=target_idx_debug,
#     k_neighbors=3,
#     max_examples=3,
#     min_score=None,
# )

# final_prompt = build_final_prompt_for_debug(sys_prompt, target_text_debug)

# print("\n" + "="*100)
# print(f"DEBUG PROMPT (target_idx={target_idx_debug}, label={lab_debug}, FEWSHOT_MODE={FEWSHOT_MODE})")
# print("="*100)
# print(final_prompt)
# print("="*100)

In [ ]:
from typing import Dict, List, Tuple, Any
import os, json, re, random
import pandas as pd

# ======================================================
# L3 STIGMA PROMPTING (FEWSHOT) — SUPPORT-STYLE FALLBACK PROTOTYPES ✅ COPY-PASTE
#
# Assumes df_in and df_pool are the SAME ROW ORDER / SAME IDS
# df_in: input posts (no Tag col needed)
# df_pool: same dataset but HAS Tag column for gold labels
# SIM_JSON_PATH maps target_idx -> [[neighbor_idx, score], ...]
#
# REQUIRED VARIABLES to set before running:
#   L3_INPUT_PATH = "...csv"
#   L3_LABELED_POOL_PATH = "...csv"
#   SIM_JSON_PATH = "...json"
#   TEXT_COL = "post_text"   (or your column)
#   TAGS_COL = "Tag"         (GT tags in df_pool)
# ======================================================

# -------------------------------
# 1) DEFINITIONS
# -------------------------------
L3_SINGLE_LABEL_DEFINITIONS: Dict[str, str] = {
   "Experienced": (
    "Any stigmatizing reactions, attitudes, or behaviors directed toward the survivor by other people. "
    "This includes direct negative responses after disclosure (blame, disbelief, minimization, shaming, dismissal), "
    "social rejection, abandonment, gossip, reputation damage, or people defending/choosing the perpetrator over the survivor. "
    "Also includes hostile stigma norms described at the group level (e.g., school, workplace, culture, online community). "
    "NOTE: Community-level stigma is merged into Experienced."
    ),
   "Internalized": (
       "Survivor expresses self-blame, shame, guilt, or feeling dirty/broken "
       "because of the assault."
   ),
   "Anticipated": (
    "Fear or expectation of being judged, blamed, disbelieved, shamed, rejected, or punished IF they disclose "
    "or seek help. This includes staying silent, avoiding reporting, or hesitating to tell others because of "
    "expected negative reactions from friends, family, authorities, institutions, or the community. "
    "The fear must be tied to anticipated social consequences of disclosure or help-seeking, rather than "
    "fear related only to physical danger, the perpetrator, trauma symptoms, or general anxiety."
  ),

   "Structural": (
       "Institutional or systemic barriers (police, hospital, university, courts, policy) "
       "that obstruct help or justice."
   ),
}

L3_LABELS: List[str] = list(L3_SINGLE_LABEL_DEFINITIONS.keys())
_CANON_L3 = {lab.lower(): lab for lab in L3_LABELS}

# -------------------------------
# Fewshot controls (like support)
# -------------------------------
FEWSHOT_MODE = "snippet"       # "full" | "truncate" | "snippet"
FEWSHOT_TRUNC_CHARS = 1200

K_NEIGHBORS = 3                # pull this many from SIM_MAP first
POS_PER_LABEL = 2              # ensure at least 2 positives in fewshot (when possible)
NEG_PER_LABEL = 1              # ensure at least 1 negative (optional but useful)
MAX_EXAMPLES = POS_PER_LABEL + NEG_PER_LABEL

RANDOM_SEED = 7
random.seed(RANDOM_SEED)

# -------------------------------
# Sanitization (optional)
# -------------------------------
def _sanitize(text: str) -> str:
    if not text:
        return ""
    s = str(text)
    repl = {
        r"\brape\b": "sexual assault",
        r"\braped\b": "sexually assaulted",
        r"\braping\b": "sexual assaulting",
        r"\bmolest(ed|ation)?\b": "sexual abuse",
        r"\bforced\b": "coerced",
        r"\bpenetrat(e|ed|ion)\b": "sexual act",
    }
    for pat, rep in repl.items():
        s = re.sub(pat, rep, s, flags=re.IGNORECASE)
    return s

# -------------------------------
# Tag parsing (case-insensitive -> canonical L3)
# -------------------------------
def parse_tags(tag_cell: Any) -> List[str]:
    if tag_cell is None:
        return []
    if isinstance(tag_cell, list):
        parts = [str(x).strip() for x in tag_cell if str(x).strip()]
    else:
        s = str(tag_cell).strip()
        if not s or s.lower() == "nan":
            return []
        parts = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]

    out = []
    for p in parts:
        key = p.lower().strip()
        if key in _CANON_L3:
            out.append(_CANON_L3[key])

    # dedupe preserve order
    seen = set()
    deduped = []
    for x in out:
        if x not in seen:
            deduped.append(x)
            seen.add(x)
    return deduped

# -------------------------------
# Snippet extraction (label-guided)
# -------------------------------
def _extract_snippet(post_text: str, lab: str, max_words: int = 28) -> str:
    text = str(post_text or "").strip()
    if not text:
        return ""

    sents = re.split(r'(?<=[.!?])\s+', text)
    sents = [s.strip() for s in sents if s.strip()]

    lab_kw = {
        "Experienced": ["blame","blamed","disbelieve","didn't believe","minimize","shame","shamed","dismiss","mock","laughed","dirty","disgust","fault","liar","lying"],
        "Internalized": ["i feel","guilty","ashamed","dirty","tainted","worthless","ruined","broken","my fault","i blame myself","i hate myself"],
        "Anticipated": ["i didn't tell","i never told","afraid","scared","worried","people would","no one would","nobody would","wouldn't believe","judge","reputation","get in trouble","make a scene"],
        "Structural": ["police","title ix","hr","hospital","court","judge","report","investigation","did nothing","refused","dismissed","ignored","no jurisdiction","lack of evidence"],
    }.get(lab, [])

    def score(sent: str) -> int:
        low = sent.lower()
        return sum(1 for k in lab_kw if k in low)

    scored = [(score(s), s) for s in sents]
    scored.sort(key=lambda x: (x[0], -len(x[1])), reverse=True)
    best = scored[0][1] if scored else text

    best = _sanitize(best)
    words = best.split()
    if len(words) > max_words:
        best = " ".join(words[:max_words]) + "…"
    return best

def _fewshot_post_for_prompt(post_text: str, lab: str) -> str:
    post_text = str(post_text or "").strip()
    if FEWSHOT_MODE == "full":
        return _sanitize(post_text)
    if FEWSHOT_MODE == "truncate":
        t = _sanitize(post_text)
        return t if len(t) <= FEWSHOT_TRUNC_CHARS else (t[:FEWSHOT_TRUNC_CHARS] + "\n...[TRUNCATED]...")
    # snippet default
    return _extract_snippet(post_text, lab, max_words=28)

# -------------------------------
# Load data
# -------------------------------
assert os.path.exists(L3_INPUT_PATH), L3_INPUT_PATH
assert os.path.exists(L3_LABELED_POOL_PATH), L3_LABELED_POOL_PATH
assert os.path.exists(SIM_JSON_PATH), SIM_JSON_PATH

df_in = pd.read_csv(L3_INPUT_PATH)
df_pool = pd.read_csv(L3_LABELED_POOL_PATH)

with open(SIM_JSON_PATH, "r") as f:
    SIM_MAP: Dict[str, List[List[Any]]] = json.load(f)

print("✅ df_in:", df_in.shape, "| df_pool:", df_pool.shape, "| SIM keys:", len(SIM_MAP))

if TEXT_COL not in df_in.columns:
    raise ValueError(f"TEXT_COL '{TEXT_COL}' not found in df_in.")
if TEXT_COL not in df_pool.columns:
    raise ValueError(f"TEXT_COL '{TEXT_COL}' not found in df_pool.")
if TAGS_COL not in df_pool.columns:
    raise ValueError(f"TAGS_COL '{TAGS_COL}' not found in df_pool (needed for fewshots).")

# -------------------------------
# Build global prototypes per label (like support)
# -------------------------------
pool_tags = df_pool[TAGS_COL].apply(parse_tags)

LABEL_TO_POS_IDXS: Dict[str, List[int]] = {lab: [] for lab in L3_LABELS}
LABEL_TO_NEG_IDXS: Dict[str, List[int]] = {lab: [] for lab in L3_LABELS}

for idx, labs in enumerate(pool_tags.tolist()):
    labs_set = set(labs)
    for lab in L3_LABELS:
        if lab in labs_set:
            LABEL_TO_POS_IDXS[lab].append(idx)
        else:
            LABEL_TO_NEG_IDXS[lab].append(idx)

# -------------------------------
# Neighbor fetch
# -------------------------------
def get_neighbor_records(target_idx: int, k: int = K_NEIGHBORS) -> List[Dict[str, Any]]:
    raw = (SIM_MAP.get(str(target_idx), []) or [])[:k]
    out = []
    for pair in raw:
        if not isinstance(pair, (list, tuple)) or len(pair) < 1:
            continue
        nb_idx = int(pair[0])
        score = float(pair[1]) if len(pair) > 1 else 1.0
        if nb_idx < 0 or nb_idx >= len(df_pool):
            continue
        nb_text = "" if pd.isna(df_pool.loc[nb_idx, TEXT_COL]) else str(df_pool.loc[nb_idx, TEXT_COL])
        nb_tags = parse_tags(df_pool.loc[nb_idx, TAGS_COL])
        out.append({"idx": nb_idx, "score": score, "text": nb_text, "tags": nb_tags})
    return out

# -------------------------------
# ✅ Support-style fewshot picker for stigma
# Ensures at least POS_PER_LABEL positives if available in pool.
# -------------------------------
def build_labelwise_fewshots_for_target(target_idx: int, lab: str) -> List[Tuple[str, str]]:
    neighbors = get_neighbor_records(target_idx, k=K_NEIGHBORS)

    pos = [n for n in neighbors if lab in set(n["tags"])]
    neg = [n for n in neighbors if lab not in set(n["tags"])]

    picked: List[Dict[str, Any]] = []

    # 1) take positives from neighbors first
    picked.extend(pos[:POS_PER_LABEL])

    # 2) if not enough positives, pull from global labeled pool positives
    if len(picked) < POS_PER_LABEL:
        need = POS_PER_LABEL - len(picked)
        candidates = [i for i in LABEL_TO_POS_IDXS.get(lab, []) if i != target_idx]
        random.shuffle(candidates)
        for i in candidates[:need]:
            txt = "" if pd.isna(df_pool.loc[i, TEXT_COL]) else str(df_pool.loc[i, TEXT_COL])
            tags = parse_tags(df_pool.loc[i, TAGS_COL])
            picked.append({"idx": i, "score": None, "text": txt, "tags": tags})

    # 3) ensure at least one negative (neighbor preferred)
    if NEG_PER_LABEL > 0:
        if neg:
            picked.extend(neg[:NEG_PER_LABEL])
        else:
            candidates = [i for i in LABEL_TO_NEG_IDXS.get(lab, []) if i != target_idx]
            random.shuffle(candidates)
            for i in candidates[:NEG_PER_LABEL]:
                txt = "" if pd.isna(df_pool.loc[i, TEXT_COL]) else str(df_pool.loc[i, TEXT_COL])
                tags = parse_tags(df_pool.loc[i, TAGS_COL])
                picked.append({"idx": i, "score": None, "text": txt, "tags": tags})

    # cap
    picked = picked[:MAX_EXAMPLES]

    # build (shown_text, answer_json)
    fewshots: List[Tuple[str, str]] = []
    for ex in picked:
        ex_text = ex["text"]
        ex_tags = ex["tags"]
        is_pos = (lab in set(ex_tags))

        shown = _fewshot_post_for_prompt(ex_text, lab)  # ✅ shown matches JSON evidence logic

        ev = _extract_snippet(ex_text, lab, max_words=20) if is_pos else ""
        ev = ev.replace("\n", " ").strip()

        ans = {
            "present": bool(is_pos),
            "evidence": ev
        }
        fewshots.append((shown, json.dumps(ans, ensure_ascii=False)))

    return fewshots

# -------------------------------
# Prompt builder
# -------------------------------
def build_l3_system_prompt(lab: str, target_idx: int) -> str:
    definition = L3_SINGLE_LABEL_DEFINITIONS[lab].strip()

    system_prompt = f"""
You are a JSON-only annotator specializing in social stigma detection.
You will evaluate ONE label at a time.

GLOBAL RULES:
- Use ONLY the post text. Do NOT infer hidden meanings.
- Decide strictly for the label under test: "{lab}".
- "present" is true ONLY if there is explicit evidence matching the label definition and boundary rules.
- The "evidence" MUST be a verbatim snippet from the post proving the label (3–25 words).
- If evidence is weak/ambiguous OR you cannot quote clear evidence, set "present": false.
- Do NOT wrap the JSON in markdown fences (no ```json).
- Return ONLY valid JSON.

LABEL UNDER TEST:
{lab}: {definition}

LABEL-SPECIFIC BOUNDARY RULES:
""".strip()

    if lab == "Experienced":
        system_prompt += """
    Experienced = stigma coming FROM OTHER PEOPLE toward the survivor.

    TRUE when there is explicit negative social reaction from others such as:
    - blame, doubt/disbelief, minimization, shaming, mocking, disgust
    - defending/siding with perpetrator
    - social rejection: avoiding, distancing, ghosting, cutting off, excluding, abandoning after learning
    - gossip/rumors/reputation damage
    - group/community stigma norms about survivors

    FALSE when:
    - it's only lack of help or emotional neglect with NO judgment/rejection signal
      (e.g., "they weren't there for me" alone is not enough)
    - it is only the survivor's self-blame/shame (Internalized)
    - the issue is institutional failure (Structural)

    Evidence must show:
    1) WHO did it (friend/family/partner/community/people online)
    2) WHAT they did/said that is stigmatizing or rejecting (a behavior or quote)
    """.strip()


    elif lab == "Internalized":
        system_prompt += """
- TRUE only if the survivor expresses first-person self-blame/shame/guilt/dirty/broken feelings.
- FALSE if blame/shame comes from other people (Experienced).
""".strip()

    elif lab == "Anticipated":
        system_prompt += """
    Anticipated (fear of stigma IF disclose/seek help):

    Set present=true ONLY if BOTH A and B are satisfied.

    A) DISCLOSURE/HELP-SEEKING HOOK (must be explicit, not just vibes)
    At least one of these must appear:
      - Disclosure/report/help actions: tell/told, disclose, speak up, open up, report, go to police, Title IX, HR, counselor, therapist,
        hotline, lawyer, court, seek help, reach out, get help
      - Silence/withholding actions: "never told anyone", "didn't tell anyone", "kept it to myself", "stayed silent",
        "can't tell anyone", "won't tell", "didn't report", "won't report", "afraid to report"
    If A is missing → present=false even if fear/anxiety is strong.

    B) EXPECTED NEGATIVE REACTION FROM OTHERS (must be social judgment/rejection/disbelief)
    At least one of these must appear:
      - judged, blamed, shamed, mocked/laughed at, disbelieved, called liar, punished, rejected, ostracized, gossip/rumors,
        "they won't believe me", "they'll think I'm lying", "they'll say it's my fault", "people will judge me",
        "they'll hate me", "I'll be kicked out", "I'll get in trouble"

    Hard FALSE rules (common false positives):
    - If fear is only about the perpetrator, retaliation, physical safety, pregnancy/STI, nightmares, flashbacks, panic, or general anxiety → present=false.
    - If they mention telling/reporting but the fear is about the PROCESS itself (court is hard, police are scary) with no social judgment/disbelief → present=false (that belongs in Structural only if there is obstruction).
    - If there is only self-blame ("I'm ashamed") with no “others will…” component → present=false (Internalized).

    Evidence requirement:
    - Evidence MUST include one snippet for A (disclose/help/silence) AND one snippet for B (expected reaction).
    If you cannot quote both → present=false.
    """.strip()



    elif lab == "Structural":
        system_prompt += """
    Structural (institution/system blocks help/justice):

    present=true ONLY if ALL 3 are present:
    1) Institution named (police, hospital, court, Title IX, HR, school admin, workplace investigation, legal system)
    2) Explicit obstruction/failure by the institution
      - refused, denied, ignored, dismissed, "did nothing", wouldn't take report, no jurisdiction, "lost my case",
        "they wouldn't investigate", "they said no evidence", "they wouldn't test", "they wouldn't help"
    3) The obstruction impacts access to help/justice (reporting, medical care, protection, accountability)

    Hard FALSE rules:
    - Institution is mentioned as an option ("I might report to police") but no obstruction happened → present=false.
    - They had a bad personal experience with a person (friend/parent/partner) that is not an institution → present=false (Experienced).
    - The barrier is purely emotional (too scared/ashamed) → present=false (Anticipated/Internalized).

    Evidence must quote the institution + the obstruction verb/phrase.
    """.strip()

    fewshots = build_labelwise_fewshots_for_target(target_idx, lab)

    system_prompt += f"""

HOW TO USE THE EXAMPLES BELOW:
- The following are gold-labeled examples similar to the one you will evaluate.
- Each example shows how the label "{lab}" can be present (true) or absent (false).
"""

    system_prompt += "\n\nFEW-SHOT EXAMPLES:\n"
    for i, (shown_post, ans_json) in enumerate(fewshots, start=1):
        system_prompt += f"""
EXAMPLE {i}
NEIGHBOR POST ({FEWSHOT_MODE}):
{shown_post}

ANSWER JSON:
{ans_json}
""".rstrip()

    system_prompt += f"""

NOW LABEL THIS POST for the "{lab}" category ONLY.

Return ONLY valid JSON in exactly this format:
{{
  "reasoning": "1–3 sentences. Name the Actor and the Action/Fear for {lab}.",
  "present": true or false,
  "evidence": "verbatim snippet if true (3–25 words), else empty string"
}}
""".strip()

    return system_prompt
# ======================================================
# DEBUG — Print the full prompt structure for one target+label ✅ COPY-PASTE
# Requires:
#   - df_in already loaded
#   - TEXT_COL defined
#   - build_l3_system_prompt(lab, target_idx) defined (from the big cell)
# ======================================================

def build_final_prompt_for_debug(system_prompt: str, target_post_text: str) -> str:
    target_post_text = "" if target_post_text is None else str(target_post_text)
    return (
        f"{system_prompt}\n\n"
        f"POST:\n{target_post_text}\n\n"
        "FINAL INSTRUCTIONS:\n"
        "Output ONLY a single JSON object (no markdown, no extra text).\n"
        "Use lowercase true/false.\n"
        "Include exactly these keys: reasoning, present, evidence.\n"
    )

# -------------------------------
# Choose what you want to inspect
# -------------------------------
target_idx_debug = 0
lab_debug = "Anticipated"   # try: "Experienced", "Internalized", "Anticipated", "Structural"

# Grab the target text
raw = df_in.loc[target_idx_debug, TEXT_COL] if (TEXT_COL in df_in.columns and target_idx_debug < len(df_in)) else ""
target_text_debug = "" if pd.isna(raw) else str(raw)

# Build prompt and print
sys_prompt = build_l3_system_prompt(lab=lab_debug, target_idx=int(target_idx_debug))
final_prompt = build_final_prompt_for_debug(sys_prompt, target_text_debug)

print("\n" + "="*110)
print(f"DEBUG PROMPT (target_idx={target_idx_debug}, label={lab_debug}, FEWSHOT_MODE={FEWSHOT_MODE})")
print("="*110)
print(final_prompt)
print("="*110)

✅ df_in: (76, 2) | df_pool: (76, 3) | SIM keys: 76

DEBUG PROMPT (target_idx=0, label=Anticipated, FEWSHOT_MODE=snippet)
You are a JSON-only annotator specializing in social stigma detection.
You will evaluate ONE label at a time.

GLOBAL RULES:
- Use ONLY the post text. Do NOT infer hidden meanings.
- Decide strictly for the label under test: "Anticipated".
- "present" is true ONLY if there is explicit evidence matching the label definition and boundary rules.
- The "evidence" MUST be a verbatim snippet from the post proving the label (3–25 words).
- If evidence is weak/ambiguous OR you cannot quote clear evidence, set "present": false.
- Do NOT wrap the JSON in markdown fences (no ```json).
- Return ONLY valid JSON.

LABEL UNDER TEST:
Anticipated: Fear or expectation of being judged, blamed, disbelieved, shamed, rejected, or punished IF they disclose or seek help. This includes staying silent, avoiding reporting, or hesitating to tell others because of expected negative reactions fro

In [ ]:
target_idx_debug = 0
print("Neighbors stored in SIM_MAP for target:", len(SIM_MAP.get(str(target_idx_debug), [])))
print("First few neighbors:", SIM_MAP.get(str(target_idx_debug), [])[:10])

Neighbors stored in SIM_MAP for target: 5
First few neighbors: [[56, 0.7013120055198669], [5, 0.7008577585220337], [62, 0.6974985003471375], [7, 0.6846855878829956], [68, 0.6800824403762817]]


In [ ]:
# ======================================================
# ✅ REPLACE YOUR WHOLE CELL WITH THIS (COPY-PASTE)
# Uses: build_l3_system_prompt(lab, target_idx, ...)
# Fixes: label list (no FINE_LABELS), non-greedy JSON extract, single-quote repair,
#        optional response_mime_type for cleaner JSON, safer blocked heuristic.
# ======================================================

from typing import Dict, Any, List
import time, json, re
import pandas as pd
from google.genai import types

MODEL_ID = "gemini-2.0-flash"

# -------------------------------
# ✅ Canonical L3 stigma labels
# -------------------------------
L3_LABELS: List[str] = list(L3_SINGLE_LABEL_DEFINITIONS.keys())  # ["Experienced","Internalized","Anticipated","Structural"]

# -------------------------------
# ✅ Gemini safety settings (lowest blocking)
# NOTE: Some content can STILL be blocked by policy.
# -------------------------------
GEMINI_SAFETY_SETTINGS = [
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
]

# If response_mime_type is not supported in your version, just remove that line.
GEMINI_CONFIG = types.GenerateContentConfig(
    temperature=0.0,
    top_p=0.95,
    max_output_tokens=600,
    safety_settings=GEMINI_SAFETY_SETTINGS,
    response_mime_type="application/json",   # ✅ BIG help if supported
)

# -------------------------------
# JSON loader (direct + extract + repairs)
# -------------------------------
def safe_json_load(txt: str) -> Dict[str, Any]:
    if not txt:
        return {}
    t = str(txt).strip()
    if not t:
        return {}

    # strip code fences
    t = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"^```\s*", "", t).strip()
    t = re.sub(r"\s*```$", "", t).strip()

    # 1) direct parse
    try:
        obj = json.loads(t)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        pass

    # 2) extract first JSON object (NON-GREEDY)
    m = re.search(r"\{.*?\}", t, flags=re.DOTALL)
    if not m:
        return {}

    blob = m.group(0).strip()

    # repair trailing commas
    blob = re.sub(r",\s*}", "}", blob)
    blob = re.sub(r",\s*]", "]", blob)

    # repair common boolean mistakes
    blob = re.sub(r"\bTrue\b", "true", blob)
    blob = re.sub(r"\bFalse\b", "false", blob)
    blob = re.sub(r"\bNone\b", "null", blob)

    # repair single-quote dicts (heuristic)
    if blob.startswith("{") and "'" in blob and '"' not in blob:
        blob = blob.replace("'", '"')

    try:
        out = json.loads(blob)
        return out if isinstance(out, dict) else {}
    except Exception:
        return {}

# -------------------------------
# Gemini response helpers
# -------------------------------
def _resp_debug_snippet(resp) -> str:
    try:
        d = resp.__dict__ if hasattr(resp, "__dict__") else {}
        return str(d)[:900]
    except Exception:
        return str(resp)[:900]

def _blocked_reason(resp) -> str:
    try:
        fb = getattr(resp, "prompt_feedback", None)
        if fb is None:
            d = resp.__dict__ if hasattr(resp, "__dict__") else {}
            fb = d.get("prompt_feedback", None)
        if fb is None:
            return "unknown"
        br = getattr(fb, "block_reason", None)
        return str(br) if br is not None else "none"
    except Exception:
        return "unknown"

def _is_blocked_response(resp) -> bool:
    """
    Some SDKs set prompt_feedback.block_reason when blocked.
    Fallback: empty text AND no candidates.
    """
    try:
        fb = getattr(resp, "prompt_feedback", None)
        if fb is None:
            d = resp.__dict__ if hasattr(resp, "__dict__") else {}
            fb = d.get("prompt_feedback", None)
        if fb is not None:
            br = getattr(fb, "block_reason", None)
            if br is not None:
                return True
    except Exception:
        pass

    txt = (getattr(resp, "text", "") or "").strip()
    if txt:
        return False

    try:
        cands = getattr(resp, "candidates", None)
        if not cands:
            return True
    except Exception:
        pass

    return False

def _sanitize_prompt_for_safety(prompt: str) -> str:
    replacements = {
        r"\brape\b": "sexual assault",
        r"\braped\b": "sexually assaulted",
        r"\braping\b": "sexual assaulting",
        r"\bmolest(ed|ation)?\b": "sexual abuse",
        r"\bsexual violence\b": "SV",
        r"\bsexual assault\b": "SA",
        r"\bpenetrat(e|ed|ion)\b": "sexual act",
        r"\bincest\b": "sexual abuse",
    }
    out = prompt
    for pat, rep in replacements.items():
        out = re.sub(pat, rep, out, flags=re.IGNORECASE)
    return out

# -------------------------------
# Core call: system_prompt + post -> JSON
# -------------------------------
def gemini_prompt_to_json(
    system_prompt: str,
    post_text: str,
    max_retries: int = 3,
    sleep_s: float = 1.2,
) -> Dict[str, Any]:

    if post_text is None or (isinstance(post_text, float) and pd.isna(post_text)):
        post_text = ""
    post_text = str(post_text)

    base_prompt = (
        f"{system_prompt}\n\n"
        f"POST:\n{post_text}\n\n"
        "FINAL INSTRUCTIONS:\n"
        "Output ONLY a single JSON object (no markdown, no extra text).\n"
        "Do NOT wrap in ``` fences.\n"
        "Use lowercase true/false.\n"
        'Include exactly these keys: "reasoning", "present", "evidence".\n'
    )

    last_err = None
    last_raw = ""

    prompts_to_try = [base_prompt, _sanitize_prompt_for_safety(base_prompt)]

    for prompt_variant_idx, prompt in enumerate(prompts_to_try, start=1):
        for attempt in range(1, max_retries + 1):
            try:
                resp = client.models.generate_content(
                    model=MODEL_ID,
                    contents=prompt,
                    config=GEMINI_CONFIG,
                )

                if _is_blocked_response(resp):
                    last_err = ValueError(
                        f"BLOCKED (variant {prompt_variant_idx}, attempt {attempt}, reason={_blocked_reason(resp)}). "
                        f"{_resp_debug_snippet(resp)}"
                    )
                    time.sleep(sleep_s)
                    continue

                txt = (getattr(resp, "text", "") or "").strip()
                last_raw = txt

                if not txt:
                    last_err = ValueError(
                        f"Empty resp.text (variant {prompt_variant_idx}, attempt {attempt}). "
                        f"Resp: {_resp_debug_snippet(resp)}"
                    )
                    time.sleep(sleep_s)
                    continue

                parsed = safe_json_load(txt)
                if not parsed:
                    last_err = ValueError(
                        f"JSON parse failed (variant {prompt_variant_idx}, attempt {attempt}). Raw head: {txt[:220]}"
                    )
                    time.sleep(sleep_s)
                    continue

                # normalize required keys
                if ("present" not in parsed) or ("evidence" not in parsed) or ("reasoning" not in parsed):
                    last_err = ValueError(
                        f"JSON missing keys (variant {prompt_variant_idx}, attempt {attempt}). Parsed: {parsed}"
                    )
                    time.sleep(sleep_s)
                    continue

                # normalize present
                if not isinstance(parsed["present"], bool):
                    p = str(parsed["present"]).strip().lower()
                    if p == "true":
                        parsed["present"] = True
                    elif p == "false":
                        parsed["present"] = False
                    else:
                        parsed["present"] = bool(parsed["present"])

                # enforce evidence empty when present=false
                if not parsed["present"]:
                    parsed["evidence"] = ""

                parsed["evidence"] = str(parsed.get("evidence", "") or "").strip()
                parsed["reasoning"] = str(parsed.get("reasoning", "") or "").strip()

                return parsed

            except Exception as e:
                last_err = e
                time.sleep(sleep_s)

    print("❌ Gemini call failed:", last_err)
    if last_raw:
        print("   Last raw model text head:", last_raw[:220])
        print("   Last raw model text tail:", last_raw[-220:])

    return {"_blocked": True, "reasoning": "blocked_or_failed", "present": False, "evidence": ""}

# ======================================================
# ✅ Level 3 runner (4 labels)
# ======================================================
def run_level3(text: str, target_idx: int) -> Dict[str, Any]:
    final_output = {
        "label_presence": {lab: False for lab in L3_LABELS},
        "labels": [],
        "evidence": {},
        "explanation": {},
        "_blocked_any": False,
        "_target_idx": int(target_idx),
    }

    if text is None or (isinstance(text, float) and pd.isna(text)):
        text = ""
    text = str(text)

    for lab in L3_LABELS:
        system_prompt = build_l3_system_prompt(
        lab=lab,
        target_idx=int(target_idx)
    )

        res = gemini_prompt_to_json(system_prompt, text)

        if res.get("_blocked"):
            final_output["_blocked_any"] = True

        is_present = bool(res.get("present", False))
        final_output["label_presence"][lab] = is_present
        final_output["explanation"][lab] = res.get("reasoning", "") or ""
        final_output["evidence"][lab] = res.get("evidence", "") or ""

        if is_present:
            final_output["labels"].append(lab)

    return final_output

def predict_level3(text: str, target_idx: int) -> Dict[str, Any]:
    try:
        return run_level3(text, target_idx)
    except Exception as e:
        print("❌ Fatal Prediction Error:", e)
        return {
            "label_presence": {k: False for k in L3_LABELS},
            "labels": [],
            "evidence": {},
            "explanation": {"error": str(e)},
            "_blocked_any": True,
            "_target_idx": int(target_idx),
        }

In [ ]:
# ==========================================
# CELL 6 — Batch predict + Progress Tracking ✅ COPY-PASTE (Improved)
# ==========================================

import json
import pandas as pd

SAVE_EVERY = 25  # checkpoint frequency

df_l3 = pd.read_csv(L3_INPUT_PATH)

if TEXT_COL not in df_l3.columns:
    raise ValueError(f"Missing TEXT_COL='{TEXT_COL}' in input CSV.")

# IMPORTANT: ensure row indices align with SIM_MAP keys (0..N-1)
df_l3 = df_l3.reset_index(drop=True)

print(f"Total posts to process: {len(df_l3)}")

# --- alignment sanity check with SIM_MAP ---
n_posts = len(df_l3)
sim_keys = set(SIM_MAP.keys())
missing = [str(i) for i in range(n_posts) if str(i) not in sim_keys]
if missing:
    print(f"⚠️ SIM_MAP missing {len(missing)} keys (first 10): {missing[:10]}")
    print("   This can cause wrong neighbor retrieval. Make sure SIM_MAP was built on the SAME df order/length.")

results = []
l3_json_list = []

for i, row in df_l3.iterrows():
    raw = row[TEXT_COL]
    text = "" if pd.isna(raw) else str(raw)

    prediction = predict_level3(text, target_idx=i)
    results.append(prediction)
    l3_json_list.append(json.dumps(prediction, ensure_ascii=False))

    # --- Print progress every 5 ---
    if (i + 1) % 5 == 0 or (i + 1) == len(df_l3):
        print(f"Processed {i + 1}/{len(df_l3)} posts...")

        labels_found = prediction.get("labels", [])
        explanations = prediction.get("explanation", {}) or {}

        print(f"   Labels Triggered: {labels_found}")

        if labels_found:
            first_lab = labels_found[0]
            reason_text = explanations.get(first_lab, "No reasoning found")
            print(f"   Model's Reasoning ({first_lab}): {str(reason_text)[:180]}...")
        else:
            # pick any label’s negative reasoning just to display something
            # (use L3_LABELS from the previous cell)
            fallback_lab = L3_LABELS[0] if "L3_LABELS" in globals() and L3_LABELS else "Experienced"
            reason_text = explanations.get(fallback_lab, "No reasoning found")
            print(f"   Model's Reasoning (Negative sample): {str(reason_text)[:180]}...")

        if prediction.get("_blocked_any", False):
            print("   ⚠️ Blocked/failed on at least one label for this post.")

        print("-" * 30)

    # --- checkpoint save ---
    if (i + 1) % SAVE_EVERY == 0:
        df_l3_tmp = df_l3.iloc[: i + 1].copy()
        df_l3_tmp["l3_json"] = l3_json_list
        tmp_path = L3_OUTPUT_PATH.replace(".csv", f"_checkpoint_{i+1}.csv")
        df_l3_tmp.to_csv(tmp_path, index=False)
        print(f"💾 Checkpoint saved: {tmp_path}")

# Save final results as JSON strings (safe for CSV)
df_l3["l3_json"] = l3_json_list

df_l3.to_csv(L3_OUTPUT_PATH, index=False)
print("✅ Saved:", L3_OUTPUT_PATH)

# Preview (only if ID_COL exists)
cols_to_show = [c for c in [ID_COL, "l3_json"] if c in df_l3.columns]
df_l3[cols_to_show].head()

Total posts to process: 76
Processed 5/76 posts...
   Labels Triggered: ['Experienced', 'Internalized']
   Model's Reasoning (Experienced): The post describes the survivor's experience with someone who pressured them into sex and made them feel used. The survivor expresses anger and resentment towards this person, indi...
------------------------------
Processed 10/76 posts...
   Labels Triggered: ['Experienced', 'Internalized', 'Anticipated']
   Model's Reasoning (Experienced): The father-in-law sexually assaulted the poster and then told her not to tell anyone. This is a negative social reaction from another person toward the survivor....
------------------------------
Processed 15/76 posts...
   Labels Triggered: ['Experienced', 'Internalized']
   Model's Reasoning (Experienced): The therapist's statement "I’m not sure how that happened. I don’t do things like that" is a minimizing and dismissive reaction to the survivor's experience, indicating a stigmatiz...
----------------------

,Post ID,l3_json
0,949vnn,"{""label_presence"": {""Experienced"": false, ""Int..."
1,jzs780,"{""label_presence"": {""Experienced"": true, ""Inte..."
2,jvot4y,"{""label_presence"": {""Experienced"": false, ""Int..."
3,nsq3ie,"{""label_presence"": {""Experienced"": true, ""Inte..."
4,c44nom,"{""label_presence"": {""Experienced"": true, ""Inte..."


In [ ]:
# ==========================================
# CELL 8 — Expand JSON into Clean Columns  ✅ COPY-PASTE (Improved)
# ==========================================
print("\nFinalizing data structure...")

import json
import re

# Use canonical labels from pipeline if available
labels_to_expand = L3_LABELS if "L3_LABELS" in globals() else ["Experienced", "Internalized", "Anticipated", "Structural"]

def _to_bool(v):
    if v is True:
        return True
    if v is False or v is None:
        return False
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"true", "yes", "y", "1"}:
            return True
        if s in {"false", "no", "n", "0", ""}:
            return False
    if isinstance(v, (int, float)):
        return bool(v)
    return False

# ------------------------------------------------------
# Helper: parse l3_json (string -> dict)
# Uses safe_json_load if available, else fallback.
# ------------------------------------------------------
def _parse_l3_json(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str) and x.strip():
        # Prefer your robust loader if defined
        if "safe_json_load" in globals():
            try:
                return safe_json_load(x) or {}
            except Exception:
                return {}
        # Fallback: plain loads
        try:
            return json.loads(x)
        except Exception:
            return {}
    return {}

df_l3["l3_json_obj"] = df_l3["l3_json"].apply(_parse_l3_json)

# ------------------------------------------------------
# 0) Coverage / blocked markers (so blocked != true negative)
# ------------------------------------------------------
def _is_blocked_any(obj):
    if not isinstance(obj, dict) or not obj:
        return True
    return _to_bool(obj.get("_blocked_any", False))

df_l3["l3_blocked_any"] = df_l3["l3_json_obj"].apply(_is_blocked_any)

df_l3["l3_covered"] = df_l3["l3_json_obj"].apply(
    lambda x: (
        isinstance(x, dict)
        and (not _to_bool(x.get("_blocked_any", False)))
        and isinstance(x.get("label_presence", None), dict)
        and all(lab in x.get("label_presence", {}) for lab in labels_to_expand)
    )
)

# ------------------------------------------------------
# 1) Expand per-label fields
# ------------------------------------------------------
for label in labels_to_expand:
    df_l3[f"{label}_present"] = df_l3["l3_json_obj"].apply(
        lambda x: _to_bool(x.get("label_presence", {}).get(label, False)) if isinstance(x, dict) else False
    )

    df_l3[f"{label}_evidence"] = df_l3["l3_json_obj"].apply(
        lambda x: str(x.get("evidence", {}).get(label, "") or "").strip() if isinstance(x, dict) else ""
    )

    df_l3[f"{label}_reasoning"] = df_l3["l3_json_obj"].apply(
        lambda x: str(x.get("explanation", {}).get(label, "") or "").strip() if isinstance(x, dict) else ""
    )

# ------------------------------------------------------
# 2) Summary labels string (comma-separated)
# ------------------------------------------------------
df_l3["l3_labels"] = df_l3["l3_json_obj"].apply(
    lambda x: ", ".join([lab for lab in (x.get("labels", []) if isinstance(x, dict) else []) if lab in labels_to_expand])
)

# ------------------------------------------------------
# 3) Save (optional: keep raw file + expanded file separate)
# ------------------------------------------------------
EXPANDED_PATH = L3_OUTPUT_PATH.replace(".csv", "_expanded.csv")

df_l3.to_csv(EXPANDED_PATH, index=False)
print(f"\nSUCCESS! Expanded results saved to: {EXPANDED_PATH}")

# ------------------------------------------------------
# 4) Useful run stats
# ------------------------------------------------------
total_with_any_label = (df_l3["l3_labels"].astype(str).str.strip() != "").sum()
print(f"Total rows with ≥1 label in this run: {int(total_with_any_label)}")

covered_n = int(df_l3["l3_covered"].sum())
blocked_n = int(df_l3["l3_blocked_any"].sum())
print(f"Coverage (unblocked & valid): {covered_n}/{len(df_l3)} = {covered_n/len(df_l3):.1%}")
print(f"Blocked/failed (any): {blocked_n}/{len(df_l3)} = {blocked_n/len(df_l3):.1%}")


Finalizing data structure...

SUCCESS! Expanded results saved to: /content/L3_predictions_expanded.csv
Total rows with ≥1 label in this run: 72
Coverage (unblocked & valid): 76/76 = 100.0%
Blocked/failed (any): 0/76 = 0.0%


In [ ]:
# =========================
# CELL A — Install/Imports ✅ COPY-PASTE (Improved)
# =========================
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

# Paths
L3_PRED_PATH = "/content/L3_predictions_expanded.csv"
L3_GT_PATH   = "/content/Sexual Violence sampled data - Level_3_annotation (1).csv"

# Columns
ID_COL = "Post ID"
GT_COL = "Tags"
PRED_COL = "l3_labels"

# Coverage flag (from Cell 8)
COVERAGE_COL = "l3_covered"

# Canonical label order (must match both GT + preds)
LABELS = ["Experienced", "Internalized", "Anticipated", "Structural"]

# --- load ---
df_pred = pd.read_csv(L3_PRED_PATH)
df_gt   = pd.read_csv(L3_GT_PATH)

# --- normalize ID types to avoid merge bugs ---
if ID_COL in df_pred.columns:
    df_pred[ID_COL] = df_pred[ID_COL].astype(str).str.strip()
if ID_COL in df_gt.columns:
    df_gt[ID_COL]   = df_gt[ID_COL].astype(str).str.strip()

# --- ensure key columns exist ---
missing_pred = [c for c in [ID_COL, PRED_COL] if c not in df_pred.columns]
if missing_pred:
    raise ValueError(f"Pred file missing columns: {missing_pred}. Found: {list(df_pred.columns)}")

if ID_COL not in df_gt.columns or GT_COL not in df_gt.columns:
    raise ValueError(f"GT file must have columns '{ID_COL}' and '{GT_COL}'. Found: {list(df_gt.columns)}")

# --- ensure coverage exists ---
if COVERAGE_COL not in df_pred.columns:
    df_pred[COVERAGE_COL] = True
    print(f"⚠️ '{COVERAGE_COL}' not found in preds. Defaulting coverage=True for all rows.")

# --- normalize text columns ---
df_pred[PRED_COL] = df_pred[PRED_COL].fillna("").astype(str)
df_gt[GT_COL]     = df_gt[GT_COL].fillna("").astype(str)

print("✅ Loaded pred rows:", len(df_pred), "| gt rows:", len(df_gt))
print("✅ Pred columns ok.")
print("✅ GT columns ok.")

# --- vocab sanity check (catches typos like 'Experience' vs 'Experienced') ---
def _split_labels(s):
    return [x.strip() for x in str(s).replace(";", ",").split(",") if x.strip()]

pred_vocab = set(l for s in df_pred[PRED_COL] for l in _split_labels(s))
gt_vocab   = set(l for s in df_gt[GT_COL] for l in _split_labels(s))

print("Pred label vocab:", sorted(pred_vocab))
print("GT label vocab:", sorted(gt_vocab))

unknown_pred = sorted(pred_vocab - set(LABELS))
unknown_gt   = sorted(gt_vocab - set(LABELS))

if unknown_pred:
    print("⚠️ Unknown labels in preds (not in LABELS):", unknown_pred)
if unknown_gt:
    print("⚠️ Unknown labels in GT (not in LABELS):", unknown_gt)

✅ Loaded pred rows: 76 | gt rows: 76
✅ Pred columns ok.
✅ GT columns ok.
Pred label vocab: ['Anticipated', 'Experienced', 'Internalized', 'Structural']
GT label vocab: ['Anticipated', 'Experienced', 'Internalized', 'Structural']


In [ ]:
# ==========================================
# CELL B — Helpers: Robust Label Parsing ✅ COPY-PASTE
# ==========================================

import ast
import pandas as pd

FINE_LABELS = LABELS  # alias

# Case-insensitive lookup map: "experienced" -> "Experienced", etc.
LABEL_MAP = {lab.lower(): lab for lab in FINE_LABELS}

def parse_labels_cell(x):
    """
    Handles:
    - Lists (['Experienced', 'Internalized'])
    - String lists ("['Experienced', 'Internalized']" / "[]")
    - Strings ("Experienced, Internalized" or "Experienced;Internalized")
    - NaNs / empty
    Normalizes by case-insensitive mapping to canonical LABELS.
    """

    # 1) None/NaN
    if x is None or pd.isna(x):
        return []

    # 2) list-like already
    if isinstance(x, list):
        items = x
    else:
        s = str(x).strip()
        if not s or s.lower() == "nan":
            return []

        # 3) if it looks like a python list string, try parsing it
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                items = parsed if isinstance(parsed, list) else [s]
            except Exception:
                items = [s]
        else:
            # 4) regular delimiter-based string
            items = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]

    out = []
    seen = set()
    for item in items:
        if item is None:
            continue
        if not isinstance(item, str):
            item = str(item)

        key = item.strip().strip('"').strip("'").strip().lower()
        if key in LABEL_MAP:
            lab = LABEL_MAP[key]
            if lab not in seen:
                out.append(lab)
                seen.add(lab)

    return out


def set_str(lbls):
    """Sort + join labels for exact-match comparisons."""
    if not lbls:
        return ""
    return ",".join(sorted(lbls))

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# CELL C — Load + Merge + Evaluate (Coverage-aware) ✅ COPY-PASTE (Improved)
# ==========================================
df_pred = pd.read_csv(L3_PRED_PATH)
df_gt   = pd.read_csv(L3_GT_PATH)

L3_LABELS = ["Experienced", "Internalized", "Anticipated", "Structural"]

# --- normalize IDs to avoid merge drops ---
df_pred[ID_COL] = df_pred[ID_COL].astype(str).str.strip()
df_gt[ID_COL]   = df_gt[ID_COL].astype(str).str.strip()

# -------------------------------
# Helper: robust bool parsing (handles True/False, 1/0, "true"/"false")
# -------------------------------
def _to_bool(v) -> bool:
    if v is None:
        return False
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, float, np.integer, np.floating)):
        if isinstance(v, float) and np.isnan(v):
            return False
        return v != 0
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"true", "t", "1", "yes", "y"}:
            return True
        if s in {"false", "f", "0", "no", "n", ""}:
            return False
        return False
    return False

# -------------------------------
# 1) Build pred_list (prefer *_present columns; fallback to l3_labels)
# -------------------------------
present_cols = [f"{lbl}_present" for lbl in L3_LABELS]
has_present_cols = all(c in df_pred.columns for c in present_cols)

def get_labels_from_presence(row):
    active = []
    for lbl in L3_LABELS:
        col = f"{lbl}_present"
        if col in row.index and _to_bool(row[col]):
            active.append(lbl)
    return active

if has_present_cols:
    df_pred["pred_list"] = df_pred.apply(get_labels_from_presence, axis=1)
else:
    if PRED_COL not in df_pred.columns:
        raise ValueError(f"Pred file missing '{PRED_COL}' and does not have *_present columns either.")
    df_pred["pred_list"] = df_pred[PRED_COL].apply(parse_labels_cell)

# -------------------------------
# 2) Merge GT + Pred
# -------------------------------
keep_cols = [ID_COL, "pred_list"]

if "l3_covered" in df_pred.columns:
    keep_cols.append("l3_covered")
if "l3_blocked_any" in df_pred.columns:
    keep_cols.append("l3_blocked_any")

merged = df_gt[[ID_COL, GT_COL]].merge(
    df_pred[keep_cols],
    on=ID_COL,
    how="inner"
)
print("Merged rows (by Post ID):", len(merged))

# -------------------------------
# 3) Parse GT labels
# -------------------------------
merged["gt_list"] = merged[GT_COL].apply(parse_labels_cell)

merged["gt_set"]   = merged["gt_list"].apply(lambda x: set(x) if isinstance(x, list) else set())
merged["pred_set"] = merged["pred_list"].apply(lambda x: set(x) if isinstance(x, list) else set())

# -------------------------------
# 4) Coverage filtering (predicted-only)
# -------------------------------
covered = merged.copy()

if "l3_covered" in covered.columns:
    covered = covered[covered["l3_covered"].apply(_to_bool)]

if "l3_blocked_any" in covered.columns:
    covered = covered[covered["l3_blocked_any"].apply(lambda v: not _to_bool(v))]

print(f"Rows used for metrics: {len(covered)}/{len(merged)} = {len(covered)/max(len(merged),1):.1%}")

# -------------------------------
# 5) Exact-match Accuracy (on covered subset)
# -------------------------------
covered["exact_match"] = (covered["gt_set"] == covered["pred_set"])
print("-" * 30)
print(f"Exact-match Accuracy (COVERED only): {covered['exact_match'].mean():.4f}")
print("-" * 30)

mismatches = covered[covered["exact_match"] == False]
if len(mismatches) > 0:
    ex = mismatches.iloc[0]
    print(f"Sample mismatch (Post ID {ex[ID_COL]}):")
    print(f"  GT:   {ex['gt_set']}")
    print(f"  PRED: {ex['pred_set']}")
else:
    print("✅ No mismatches found (perfect exact match on covered rows).")

# -------------------------------
# 6) Multilabel PRF (micro/macro + per-label)
# -------------------------------
mlb = MultiLabelBinarizer(classes=L3_LABELS)
y_true = mlb.fit_transform(covered["gt_list"])
y_pred = mlb.transform(covered["pred_list"])

print("\nClassification report (covered only):")
print(classification_report(y_true, y_pred, target_names=L3_LABELS, zero_division=0))

p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)

print(f"Micro  P/R/F1: {p_micro:.3f} / {r_micro:.3f} / {f_micro:.3f}")
print(f"Macro  P/R/F1: {p_macro:.3f} / {r_macro:.3f} / {f_macro:.3f}")

Merged rows (by Post ID): 76
Rows used for metrics: 76/76 = 100.0%
------------------------------
Exact-match Accuracy (COVERED only): 0.3947
------------------------------
Sample mismatch (Post ID jzs780):
  GT:   {'Anticipated', 'Internalized'}
  PRED: {'Anticipated', 'Internalized', 'Experienced'}

Classification report (covered only):
              precision    recall  f1-score   support

 Experienced       0.66      0.90      0.76        41
Internalized       0.85      0.96      0.91        55
 Anticipated       0.59      0.52      0.55        25
  Structural       1.00      0.55      0.71        11

   micro avg       0.75      0.83      0.78       132
   macro avg       0.78      0.73      0.73       132
weighted avg       0.76      0.83      0.78       132
 samples avg       0.74      0.83      0.75       132

Micro  P/R/F1: 0.747 / 0.826 / 0.784
Macro  P/R/F1: 0.777 / 0.733 / 0.732


In [ ]:
print(merged['gt_set'].iloc[0], merged['pred_set'].iloc[0])

{'Internalized'} {'Internalized'}


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np

FINE_LABELS = LABELS
mlb = MultiLabelBinarizer(classes=FINE_LABELS)

LABEL_MAP = {lab.lower(): lab for lab in FINE_LABELS}

def split_labels(x):
    """
    Robust multilabel parser.

    Accepts:
    - list: ['Experienced', 'Anticipated']
    - string: "Experienced, Anticipated"
    - NaN / empty

    Returns:
    - canonical labels
    - ordered according to FINE_LABELS
    """

    # -------------------
    # Normalize input
    # -------------------
    if isinstance(x, list):
        parts = [str(p).strip() for p in x if isinstance(p, str)]

    elif x is None or (isinstance(x, float) and np.isnan(x)):
        return []

    else:
        s = str(x).strip()
        if not s or s.lower() == "nan":
            return []
        parts = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]

    # -------------------
    # Canonical mapping
    # -------------------
    mapped = []
    unknown = []

    for p in parts:
        key = p.lower()
        if key in LABEL_MAP:
            mapped.append(LABEL_MAP[key])
        else:
            unknown.append(p)

    # Optional debug warning (VERY useful once)
    if unknown:
        print(f"⚠️ Unknown labels ignored: {unknown}")

    # -------------------
    # Deduplicate + canonical order
    # -------------------
    mapped_set = set(mapped)
    ordered = [lab for lab in FINE_LABELS if lab in mapped_set]

    return ordered

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report
import pandas as pd

# ✅ Use COVERED subset if available, else fallback to merged
df_eval = covered if "covered" in globals() else merged
df_eval = df_eval.copy()  # avoid SettingWithCopy issues

# Use ONE robust parser for both GT and preds
df_eval["gt_list_clean"] = df_eval["gt_list"].apply(split_labels)       # GT
df_eval["pred_list_clean"] = df_eval["pred_list"].apply(split_labels)   # Pred

# Lock class order explicitly
mlb = MultiLabelBinarizer(classes=FINE_LABELS)
mlb.fit([[]])  # initializes internal structures

y_true = mlb.transform(df_eval["gt_list_clean"])
y_pred = mlb.transform(df_eval["pred_list_clean"])

print("Classification Report for Level 3 Stigma Labels (COVERED only):")
print("-" * 60)
print(classification_report(y_true, y_pred, target_names=FINE_LABELS, zero_division=0))
print("-" * 60)

report_dict = classification_report(
    y_true, y_pred,
    target_names=FINE_LABELS,
    output_dict=True,
    zero_division=0
)

df_report = pd.DataFrame(report_dict).transpose()
df_report

Classification Report for Level 3 Stigma Labels (COVERED only):
------------------------------------------------------------
              precision    recall  f1-score   support

 Experienced       0.66      0.90      0.76        41
Internalized       0.85      0.96      0.91        55
 Anticipated       0.59      0.52      0.55        25
  Structural       1.00      0.55      0.71        11

   micro avg       0.75      0.83      0.78       132
   macro avg       0.78      0.73      0.73       132
weighted avg       0.76      0.83      0.78       132
 samples avg       0.74      0.83      0.75       132

------------------------------------------------------------


,precision,recall,f1-score,support
Experienced,0.660714,0.902439,0.762887,41.0
Internalized,0.854839,0.963636,0.905983,55.0
Anticipated,0.590909,0.520000,0.553191,25.0
Structural,1.000000,0.545455,0.705882,11.0
micro avg,0.746575,0.825758,0.784173,132.0
macro avg,0.776616,0.732882,0.731986,132.0
weighted avg,0.756653,0.825758,0.778045,132.0
samples avg,0.736842,0.826754,0.752381,132.0


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="micro",
    labels=range(len(FINE_LABELS)),
    zero_division=0
)

print("LEVEL 3 (Micro)")
print("Precision:", round(p_micro, 4))
print("Recall:   ", round(r_micro, 4))
print("F1:       ", round(f1_micro, 4))


LEVEL 3 (Micro)
Precision: 0.7466
Recall:    0.8258
F1:        0.7842


In [ ]:
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    labels=range(len(FINE_LABELS)),
    zero_division=0
)

print("\nLEVEL 3 (Macro)")
print("Precision:", round(p_macro, 4))
print("Recall:   ", round(r_macro, 4))
print("F1:       ", round(f1_macro, 4))


LEVEL 3 (Macro)
Precision: 0.7766
Recall:    0.7329
F1:        0.732


In [ ]:
p_lbl, r_lbl, f1_lbl, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

for i, label in enumerate(FINE_LABELS):
    print(f"{label:12s} | P: {p_lbl[i]:.3f} | R: {r_lbl[i]:.3f} | F1: {f1_lbl[i]:.3f} | Support: {support[i]}")

Experienced  | P: 0.661 | R: 0.902 | F1: 0.763 | Support: 41
Internalized | P: 0.855 | R: 0.964 | F1: 0.906 | Support: 55
Anticipated  | P: 0.591 | R: 0.520 | F1: 0.553 | Support: 25
Structural   | P: 1.000 | R: 0.545 | F1: 0.706 | Support: 11


In [ ]:
# Show unique raw GT tag strings (sample)
df_gt[GT_COL].astype(str).str.strip().value_counts().head(30)

,count
Tags,
Internalized,18
"Internalized, Experienced",12
Experienced,9
"Internalized, Experienced, Anticipated",6
"Anticipated, Internalized",6
"Experienced, Internalized",5
"Internalized, Anticipated",4
Structural,3
"Structural, Experienced",2


In [ ]:
print("len(merged):", len(merged))
print("len(covered):", len(covered) if "covered" in globals() else None)

# If you evaluate on "covered", lock that in:
df_eval = covered.copy()
print("Evaluating rows:", len(df_eval))

# GT support counts inside eval subset
for lab in LABELS:
    c = df_eval["gt_list"].apply(lambda x: lab in set(x)).sum()
    print(lab, int(c))

len(merged): 76
len(covered): 76
Evaluating rows: 76
Experienced 41
Internalized 55
Anticipated 25
Structural 11


In [ ]:
df_in = df_in.reset_index(drop=True)
df_pool = df_pool.reset_index(drop=True)

In [ ]:
t = 0
print("TARGET:", df_in.loc[t, TEXT_COL][:200], "...\n")
for nb in get_neighbor_records(t, k=5):
    print("NB idx:", nb["pool_idx"], "score:", nb["score"])
    print("NB txt:", nb["text"][:180], "...\n")

TARGET: Last year I was on a university first year camp, and I had sex with a guy while I was very drunk. I had been chatting/flirting with him but had decided that I didn't want to do anything with him, much ...

NB idx: 56 score: 0.7013120055198669
NB txt: over a year ago, i met a guy i thought was super cool on tinder. we made plans to hang out at the pool together. i didn’t have a car at the time so he had to pick me up. 
  
  
 

 ...

NB idx: 5 score: 0.7008577585220337
NB txt: Some months ago I had sex for the first time. I was really nervous but I want to make it clear that at no point did I say no, though in retrospect I’m mad at myself for trapping my ...

NB idx: 62 score: 0.6974985003471375
NB txt: Tldr and trigger warning: had sex I didn't want but did ask for. Everyone involved was hurt, and it's possible a child saw. 
  
  
 

  I suffer from dissociation/derealization tha ...

